In [ ]:
import yaml
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact


In [ ]:
# Loading data
with open ('config.yaml', 'r') as file_adress:
    config = yaml.safe_load(file_adress)

RAfile = config['resultmonster']
df = pd.read_csv(RAfile)  


In [ ]:
#inspecting first 5 rows
df.head() 

In [ ]:
def filter_dataframe_by_farmer_name(df, farmer_name):
    """
    :df is the data frame we want to get the subset from
    :farmer_name is the farmer we want the df to be filtered for
    :returns a new dataframe that contains only the rows where the 'plot' column contains the farmer name.
    """
    return df[df['plot'].str.contains(farmer_name)]

In [ ]:
# Making farmer subset
farmer_df = filter_dataframe_by_farmer_name(df,'Ossedijk') 

In [ ]:
# check for missing values in the data frame
missing_values = farmer_df.isnull().sum()
missing_values

In [ ]:
farmer_df.describe( )

In [ ]:
corr_matrix = farmer_df.corr()
# plot a heatmap of the correlation matrix
sns.heatmap(corr_matrix, cmap = 'coolwarm' ) 
plt.show()

In [ ]:
def DS_Q_Q_Plot(y, est = 'robust', **kwargs):
    """
    *
    Function DS_Q_Q_Plot(y, est = 'robust', **kwargs)
    
       This function makes a normal quantile-quantile plot (Q-Q-plot), also known
       as a probability plot, to visually check whether data follow a normal distribution.
    
    Requires:            - 
    
    Arguments:
      y                  data array
      est                Estimation method for normal parameters mu and sigma:
                         either 'robust' (default), or 'ML' (Maximum Likelihood),
                         or 'preset' (given values)
      N.B. If est='preset' than the *optional* parameters mu, sigma must be provided:
      mu                 preset value of mu
      sigma              preset value of sigma
      
    Returns:
      Estimated mu, sigma, n, and expected number of datapoints outside CI in Q-Q-plot.
      Q-Q-plot
      
    Author:            M.E.F. Apol
    Date:              2020-01-06, revision 2022-08-30
    """
    
    import numpy as np
    from scipy.stats import iqr # iqr is the Interquartile Range function
    import matplotlib.pyplot as plt
    
    y = farmer_df[y]
    # First, get the optional arguments mu and sigma:
    mu_0 = kwargs.get('mu', None)
    sigma_0 = kwargs.get('sigma', None)
    
    n = len(y)
    
    # Calculate order statistic:
    y_os = np.sort(y)
  
    # Estimates of mu and sigma:
    # ML estimates:
    mu_ML = np.mean(y)
    sigma2_ML = np.var(y)
    sigma_ML = np.std(y) # biased estimate
    s2 = np.var(y, ddof=1)
    s = np.std(y, ddof=1) # unbiased estimate
    # Robust estimates:
    mu_R = np.median(y)
    sigma_R = iqr(y)/1.349

    # Assign values of mu and sigma for z-transform:
    if est == 'ML':
        mu, sigma = mu_ML, s
    elif est == 'robust':
        mu, sigma = mu_R, sigma_R
    elif est == 'preset':
        mu, sigma = mu_0, sigma_0
    else:
        print('Wrong estimation method chosen!')
        return()
        
    print('Estimation method: ' + est)
    print('n = {:d}, mu = {:.4g}, sigma = {:.4g}'.format(n, mu,sigma))
    
    # Expected number of deviations (95% confidence level):
    n_dev = np.round(0.05*n)
    
    print('Expected number of data outside CI: {:.0f}'.format(n_dev))
         
    # Perform z-transform: sample quantiles z.i
    z_i = (y_os - mu)/sigma

    # Calculate cumulative probabilities p.i:
    i = np.array(range(n)) + 1
    p_i = (i - 0.5)/n

    # Calculate theoretical quantiles z.(i):
    from scipy.stats import norm
    z_th = norm.ppf(p_i, 0, 1)

    # Calculate SE or theoretical quantiles:
    SE_z_th = (1/norm.pdf(z_th, 0, 1)) * np.sqrt((p_i * (1 - p_i)) / n)

    # Calculate 95% CI of diagonal line:
    CI_upper = z_th + 1.96 * SE_z_th
    CI_lower = z_th - 1.96 * SE_z_th

    # Make Q-Q plot:
    plt.plot(z_th, z_i, 'o', color='k', label='experimental data')
    plt.plot(z_th, z_th, '--', color='r', label='normal line')
    plt.plot(z_th, CI_upper, '--', color='b', label='95% CI')
    plt.plot(z_th, CI_lower, '--', color='b')
    plt.xlabel('Theoretical quantiles, $z_{(i)}$')
    plt.ylabel('Sample quantiles, $z_i$')
    plt.title('Q-Q plot (' + est + ')')
    plt.legend(loc='best')
    plt.show()
    pass;

In [ ]:
# Selecting numeric columns for plotting purpose
numeric_cols = farmer_df.select_dtypes(include=['int', 'float']).columns 
numeric_cols = [col for col in numeric_cols if pd.to_numeric(farmer_df[col], errors='coerce').notnull().all()] 

In [ ]:
 
# create a dropdown widget for selecting the column name
column_selector = widgets.Dropdown(options = numeric_cols, description = 'Column') 

# create the interactive panel
interact(DS_Q_Q_Plot, y = column_selector)
